# Detection defences under the σ attack — DLA, DNN+, ICNN and PAD

The second defence family does not harden the classifier; it **detects** adversarial inputs and rejects them. Four detectors from the literature are measured: **DLA**, **DNN+**, **ICNN** and **PAD**. Each is checked the same way — its threshold (τ), its behaviour on benign, malware and slightly perturbed inputs — then attacked with σ twice: with the attack *oblivious* to the detector (oblivion=True) and with the attack aware of it (oblivion=False), because a detector only counts if it survives an attacker who knows it is there.

**Where this fits:** Section 03 built the attacks; this section asks how the **defences** hold up against them — above all against the σ attack that grew out of `03-attacks/cw_sine_lineage.ipynb`. The same defences, measured in the final organised protocol, are `06-thesis-evaluation/`.

*Consolidated from `DNN&DLA&ICNN&DNN+.ipynb`, `ICNN.ipynb`, `DLA.ipynb` and `DNNPlus.ipynb` (originally in `PGD_form_attacks`); the single-model notebooks' runs duplicate the combined notebook and are not repeated — their unique construction code is folded in. Outputs cleared.*

## Step 1 — Setup: data, the manipulation mask, and the shared scoring harness

# Loading data

In [ ]:
import torch
import torch.nn as nn

# Assuming you are using a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown

download_links = [
                  'https://drive.google.com/uc?id=1dkoFDyJMuGewWX-LE5rb7zkRU6XkSa2z',
                  'https://drive.google.com/uc?id=1oHJBbqyKybmlFFRZ7-uftn77Vg0Hr_A5',
                  'https://drive.google.com/uc?id=1YwxR5CCNnG5gjeFBlBA8mEfDQM22mLj_',
                  'https://drive.google.com/uc?id=1XB5jdefjnGSxrb27SO-65JZpz5UcsNh2',
                  'https://drive.google.com/uc?id=1Ahw4prIhm322LU53Yhkju5ixd6q1KCJ6',
                  'https://drive.google.com/uc?id=18_YhP6qHHmu52CfANYD2LqxrOzq5ECxl',
                  'https://drive.google.com/uc?id=140wKHyRpKurSuvEOswKdyH3kKRPg8VW_',
                  'https://drive.google.com/uc?id=1pTokv8xKvZoUo8kejJc7N757iNfHmt2s',
                  'https://drive.google.com/uc?id=107sDEWVIJZJWC1SIWASEepdRXB3VWlCZ',
                  'https://drive.google.com/uc?id=1sFBZ3A8VQ-iDjmwASg4u5uHQQR8kVJy9',


]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import zipfile

# Define the path to the zip file and the extraction path
zip_file_path = '/content/datasets.zip'
extract_to_path = '/content/'

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tensors
train_features = torch.load('train_features.pt', weights_only=True).float()
train_labels = torch.load('train_labels.pt', weights_only=True).long()
val_features = torch.load('val_features.pt', weights_only=True).float()
val_labels = torch.load('val_labels.pt', weights_only=True).long()
test_features = torch.load('test_features.pt', weights_only=True).float()
test_labels = torch.load('test_labels.pt', weights_only=True).long()

# Print data shapes for verification
print(f"Train Features Shape: {train_features.shape}")
print(f"Train Labels Shape: {train_labels.shape}")
print(f"Validation Features Shape: {val_features.shape}")
print(f"Validation Labels Shape: {val_labels.shape}")
print(f"Test Features Shape: {test_features.shape}")
print(f"Test Labels Shape: {test_labels.shape}")

# Create datasets
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

# Set batch size
batch_size = 10000
# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Verify DataLoader sizes
print('----------------------------------------------')
print(f"Train Loader Batch Count: {len(train_loader)}")
print(f"Validation Loader Batch Count: {len(val_loader)}")
print(f"Test Loader Batch Count: {len(test_loader)}")

In [ ]:
# Function to count samples for each label
def count_labels(labels):
    count_0 = (labels == 0).sum().item()  # Count samples with label = 0
    count_1 = (labels == 1).sum().item()  # Count samples with label = 1
    return count_0, count_1

# Count labels for each dataset
train_count_0, train_count_1 = count_labels(train_labels)
val_count_0, val_count_1 = count_labels(val_labels)
test_count_0, test_count_1 = count_labels(test_labels)

# Calculate total counts across all datasets
total_count_0 = train_count_0 + val_count_0 + test_count_0
total_count_1 = train_count_1 + val_count_1 + test_count_1

# Print results
print(f"Training set             \t Label 0: {train_count_0}\t Label 1: {train_count_1}")
print(f"Validation set           \t Label 0: {val_count_0}\t Label 1: {val_count_1}")
print(f"Test set                 \t Label 0: {test_count_0}\t Label 1: {test_count_1}")
print(f"Total across all datasets\t Label 0: {total_count_0}\t Label 1: {total_count_1}")

In [ ]:
print(f"The avrage number of feature in Training set: {train_features.sum()/train_features.shape[0]:.2f}")
print(f"The avrage number of feature in Validation set: {val_features.sum()/val_features.shape[0]:.2f}")
print(f"The avrage number of feature in Test set: {test_features.sum()/test_features.shape[0]:.2f}")

In [ ]:
removal_array = torch.load('manipulation_x.pt', weights_only=True, map_location=device)
removal_array.sum()

In [ ]:
import torch

def final_rounding(oimgs, newimg, model, removal_array, is_detector_enabled=False, oblivion=False):
    """
    Apply final rounding adjustments to adversarial images based on the model's predictions.

    Parameters:
    - oimgs (Tensor): Original images tensor.
    - newimg (Tensor): Adversarially perturbed images tensor.
    - model (nn.Module): The model used to evaluate active samples.
    - removal_array (Tensor): Array indicating features that should not be modified.

    Returns:
    - final (Tensor): The adjusted adversarial images.
    """

    # Ensure tensors are on the correct device
    device = oimgs.device
    oimgs, perturbed, removal_array = oimgs.to(device), newimg.to(device), removal_array.to(device)

    # Create mask for non-fixed features as a boolean tensor
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).bool()

    # Set the model to evaluation mode
    model.eval()

    # Calculate differences and prepare significant change mask
    with torch.no_grad():

        if is_detector_enabled:
            if model.name == 'kde':
                logits, x_hidden = model.forward_f(oimgs)
                prob = model.forward_g(x_hidden, y_pred=0)
            else:
                logits, prob = model.forward(oimgs)
        else:
            logits = model.forward(oimgs)

        if is_detector_enabled and not oblivion:
            tau = model.tau[0].item()
            inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
        else:
            inactive_mask = (logits.argmax(dim=1) == 0)

        active_mask = ~inactive_mask

        # Calculate total change for each sample
        total_change = (perturbed - oimgs).abs()
        significant_mask = total_change > 0.1
        sorted_idx = total_change.argsort(dim=1, descending=True)

        # Define changes to be applied, ensuring correct type and device
        changes = torch.where(perturbed > oimgs, 1, torch.where(perturbed < oimgs, 0, oimgs)).float().to(device)

    # Clone the original images for final adjustments
    final = oimgs.clone()

    # Set a practical maximum for significant changes per sample
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 200)

    # Iterate through each significant change index
    for i in range(max_significant_changes):
        if not active_mask.any():
            break

        active_indices = torch.where(active_mask)[0]

        # Update the final image with changes based on non-fixed feature mask
        indices = sorted_idx[active_indices, i]
        final[active_indices, indices] = torch.where(
            non_fixed_features_mask[active_indices, indices],
            changes[active_indices, indices],
            final[active_indices, indices]
        )

        # Update active_mask based on model prediction after applying changes
        with torch.no_grad():

            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(final[active_indices])
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(final[active_indices])
            else:
                logits = model.forward(final[active_indices])

            if is_detector_enabled and not oblivion:
                inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
            else:
                inactive_mask = (logits.argmax(dim=1) == 0)

            active_mask[active_indices] = (~inactive_mask)

    return final

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")

    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")


def plot_adv_accuracies(adv_rounded_list, labels, model, mal_x_list, is_detector_enabled=False, device='cpu', batch_size=512, norm='L2', interval=5, oblivion=False):
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Disable LaTeX rendering to avoid errors
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(18, 12))
    model.eval()
    x_min, x_max = 0, 0

    for idx, (adv_rounded, mal_x, color, label) in tqdm(enumerate(zip(adv_rounded_list, mal_x_list, colors, labels)),
                                                  total=len(adv_rounded_list), desc="Processing Tensors"):
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if is_detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances)
            sorted_distances, _ = torch.sort(total_distances)
            x_values = sorted_distances.cpu()

            x_min = min(x_min, int(x_values.min().item()))
            x_max = max(x_max, int(x_values.max().item()))

            # Calculate percentage values for accuracy on y-axis
            y_values = (1 - torch.arange(1, len(sorted_distances) + 1).float().cpu() / num_samples) * 100

            sns.lineplot(x=x_values, y=y_values, marker=markers[idx % len(markers)], color=color,
                         linestyle=line_styles[idx % len(line_styles)], linewidth=2, markersize=5, alpha=0.8, label=label)

    if is_detector_enabled:
        if oblivion:
            attack_type = 'oblivious attack'
        else:
            attack_type = 'adaptive attack'
    else:
        attack_type = 'attack'

    plt.xlabel(f'Distance ({norm} norm)', fontsize=14, labelpad=10)
    plt.ylabel('Model Accuracy (%)', fontsize=14, labelpad=10)
    #plt.title('Model Accuracy vs. Distance from Adversarial Examples', fontsize=16, pad=10)
    plt.title(f'Accuracy (%) of detector {model.name} vs. Distance under {attack_type}', fontsize=16, pad=10)
    plt.legend(title='Adversarial Sets', fontsize=10, title_fontsize=12, loc='upper right')

    # Set x-axis with integer values but fewer ticks
    plt.grid(True, which='major', linestyle='--', linewidth=0.5)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.5)

    # Y-axis ticks for percentage values, every 10% from 0% to 100%
    plt.yticks(ticks=range(0, 110, 10))

    # Set x-axis ticks at intervals, e.g., every 5 units
    plt.xticks(ticks=range(x_min, x_max + 1, interval))

    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from typing import Any, Tuple

def attack_mal_batches(
    test_loader: DataLoader,
    model: nn.Module,
    attack: Any,
    removal_array: torch.Tensor,
    device: torch.device,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    indicator_masking: bool = False,
    **kwargs
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Applies adversarial attacks on malicious samples within batches and returns
    the original malicious samples and the corresponding adversarial examples.

    Args:
        test_loader (DataLoader): DataLoader for test data.
        model (nn.Module): Model to be attacked.
        attack (Any): Adversarial attack function (e.g., sigma_zero_rounded).
        removal_array (torch.Tensor): Masking tensor indicating removable features.
        device (torch.device): Device for computation.
        is_detector_enabled (bool, optional): Enables/disables detector during attack.
        oblivion (bool, optional): Flag for attack configuration.
        indicator_masking (bool, optional): Enables indicator masking for suspecious samples (prob>tau).
        **kwargs: Additional parameters for the attack function (e.g., steps, lr).

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: Original malicious samples and adversarial examples.
    """
    model.eval()
    perturbed_batches = []
    perturbed_indicator_batches = []
    mal_x_batches = []

    # Iterate through each batch in the test_loader
    for x_test, y_test in test_loader:
        torch.cuda.empty_cache()

        # Move batch to device
        x_test, y_test = x_test.to(device), y_test.to(device)

        # Select malicious samples (label == 1)
        mal_x_batch = x_test[y_test.squeeze() == 1]

        if mal_x_batch.size(0) > 0:  # Process only if there are malicious samples
            # Generate adversarial examples for malicious samples
            perturbed_batch = attack(
                model=model,
                inputs=mal_x_batch,
                removal_array=removal_array,
                device=device,
                is_detector_enabled=is_detector_enabled,
                oblivion=oblivion,
                **kwargs
            )
            perturbed_batches.append(perturbed_batch)
            mal_x_batches.append(mal_x_batch)

            # Apply indicator masking if enabled and required
            if is_detector_enabled and not oblivion and indicator_masking:
                print(f"{'-'*50}")
                print('Oblivion Attack ')
                perturbed_indicator_batch = attack(
                    model=model,
                    inputs=mal_x_batch,
                    removal_array=removal_array,
                    device=device,
                    is_detector_enabled=is_detector_enabled,
                    oblivion=True,
                    **kwargs
                )
                perturbed_indicator_batches.append(perturbed_indicator_batch)

    # Combine all malicious samples and their perturbed counterparts
    combined_mals = torch.cat(mal_x_batches, dim=0)
    combined_advs = torch.cat(perturbed_batches, dim=0)

    # Initialize final adversarial examples based on attack conditions
    if is_detector_enabled and not oblivion and indicator_masking:
        combined_indicator_advs = torch.cat(perturbed_indicator_batches, dim=0)
        final_advs = combined_advs.clone()

        # Get logits and probability
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(final_advs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model(final_advs)

        tau = model.tau[0].item()
        success_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)

        # Update unsuccessful adversarial examples using the indicator batch
        final_advs = torch.where(success_mask.unsqueeze(-1), final_advs, combined_indicator_advs)
    else:
        final_advs = combined_advs

    return combined_mals, final_advs

In [ ]:
import torch
from torch import nn
from typing import List

def calculate_distance(adv_samples: torch.Tensor, original_samples: torch.Tensor, norm: str) -> torch.Tensor:
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Args:
        adv_samples (torch.Tensor): The adversarial samples.
        original_samples (torch.Tensor): The original samples.
        norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: Calculated distances between each adversarial and original sample.
    """
    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")

def sum_of_top_n_distances(adv_rounded: torch.Tensor, mal_x_batch: torch.Tensor, y_pred: torch.Tensor, n: int, norm: str = 'L1') -> torch.Tensor:
    """
    Calculate the sum of the top n smallest distances between adv_rounded and mal_x_batch
    for samples where y_pred == 0, based on the specified norm.

    Args:
        adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
        mal_x_batch (torch.Tensor): The original malicious examples tensor.
        y_pred (torch.Tensor): The prediction tensor.
        n (int): The number of top distances to sum.
        norm (str): The norm to use ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: The sum of the top n smallest distances.
    """
    distances = calculate_distance(adv_rounded[y_pred == 0], mal_x_batch[y_pred == 0], norm)
    top_n_distances = torch.topk(distances, n, largest=False).values
    return top_n_distances.sum()

def evaluate_adversarial_examples(
    mal_x: torch.Tensor,
    adv: torch.Tensor,
    model: nn.Module,
    removal_array: torch.Tensor,
    is_detector_enabled: bool = False,
    oblivion: bool = False,
    indicator_masking: bool = False,
    rounded: bool = False,
    batch_size: int = 1024,
    norm: str = 'L1'
) -> torch.Tensor:
    """
    Evaluates adversarial examples against a model, calculating various distance metrics
    and success rates for different norms, optionally in batches.

    Args:
        mal_x (torch.Tensor): Original malware input.
        adv (torch.Tensor): Adversarial examples generated from mal_x.
        model (nn.Module): The trained model to be evaluated.
        removal_array (torch.Tensor): Array used for final rounding.
        is_detector_enabled (bool, optional): Enables detector processing during evaluation.
        oblivion (bool, optional): Alters attack configuration if true.
        indicator_masking (bool, optional): Enables indicator masking for suspect samples.
        rounded (bool, optional): Applies final rounding on adversarial examples.
        batch_size (int, optional): Maximum samples per batch for processing.
        norm (str): Norm used for distance calculations ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: The final adversarial examples used in evaluation.
    """
    model.eval()
    torch.cuda.empty_cache()

    total_samples = mal_x.shape[0]
    num_successful_attacks = 0
    total_distances = 0
    y_pred_list = []
    mal_x_list = []
    final_adv_list = []
    all_distances: List[float] = []

    for i in range(0, total_samples, batch_size):
        mal_x_batch = mal_x[i:i + batch_size]
        adv_batch = adv[i:i + batch_size]

        # Apply rounding if specified
        final_adv_batch = (
            final_rounding(mal_x_batch, adv_batch, model, removal_array, is_detector_enabled, oblivion)
            if rounded else adv_batch.clone()
        )

        # Perform forward pass and determine successful attacks
        with torch.no_grad():
            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(final_adv_batch)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model(final_adv_batch)

                # Identify successful attacks
                successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())

                # Exclude samples for indicator masking
                if indicator_masking:
                    exclusion_mask = (logits.argmax(dim=1) == 0) & (prob > model.tau[0].item())
                    successful_attacks_mask = successful_attacks_mask[~exclusion_mask]
                    mal_x_batch = mal_x_batch[~exclusion_mask]
                    final_adv_batch = final_adv_batch[~exclusion_mask]

            else:
                logits = model(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0)

        final_adv_list.append(final_adv_batch)
        y_pred_batch = (~successful_attacks_mask).long()
        y_pred_list.append(y_pred_batch)
        mal_x_list.append(mal_x_batch)

        # Calculate distances for successful attacks
        num_successful_attacks += successful_attacks_mask.sum().item()
        if successful_attacks_mask.any():
            distances = calculate_distance(final_adv_batch[successful_attacks_mask], mal_x_batch[successful_attacks_mask], norm)
            total_distances += distances.sum().item()
            all_distances.extend(distances.cpu().tolist())

    # Aggregate predictions and adversarial examples
    y_pred = torch.cat(y_pred_list)
    final_adv = torch.cat(final_adv_list)
    final_mal_x = torch.cat(mal_x_list)

    # Calculate metrics
    num_considered_malwares = len(y_pred)
    accuracy = (y_pred == 1).float().mean().item() * 100
    mean_distance = total_distances / max(1, num_successful_attacks)
    median_distance = torch.median(torch.tensor(all_distances)).item() if all_distances else 0

    # Print results
    print(f"Evaluation on Adversarial Examples (Norm: {norm}, Rounded: {rounded})")
    print("-----------------------------------------------")
    print(f"Total malwares: {total_samples} and Considered malwares : {num_considered_malwares}")
    print(f"Successful attacks: {num_successful_attacks}/{num_considered_malwares}")
    print(f"Model accuracy: {accuracy:.2f}%")
    print(f"Mean distance (successful): {mean_distance:.2f}")
    print(f"Median distance (successful): {median_distance:.2f}")

    # Calculate top N distances
    top_n_values = [500, 1000, 2000, 2500, 3000]
    for top_n in top_n_values:
        if num_successful_attacks >= top_n:
            mean_top_n_distance = sum_of_top_n_distances(final_adv, final_mal_x, y_pred, top_n, norm=norm) / top_n
            print(f"Mean distance (TOP {top_n} successful): {mean_top_n_distance:.2f}")

    return final_adv, final_mal_x

In [ ]:
import torch
import gzip
import zipfile
import os


def save_model_data_as_zip(
    model_name,
    output_dir="output",
    is_detector_enabled=False,
    indicator_masking=False,
    **tensors,
):
    """
    Save model data tensors as compressed binary files and package them into a zip file.

    Args:
        model_name (str): Name of the model (e.g., "ICNN").
        output_dir (str): Directory to save the compressed files and the zip.
        is_detector_enabled (bool): Whether the detector is enabled.
        indicator_masking (bool): Whether indicator masking is enabled.
        **tensors: Keyword arguments for tensors (e.g., mals, advs, mals_detect, etc.).
                   Provide tensors dynamically depending on the configuration.

    Required Tensors:
        - If `is_detector_enabled` is False:
            * mals
            * advs
            * advs_r
        - If `is_detector_enabled` is True:
            * mals_detect
            * advs_detect
            * advs_detect_r
            * advs_obliv
            * advs_obliv_r
            * (Optional) advs_indic, advs_indic_r (if `indicator_masking` is True)
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Prepare the tensor dictionary based on configurations
    if is_detector_enabled:
        required_keys = ["mals_detect", "advs_detect", "advs_detect_r", "advs_obliv", "advs_obliv_r"]
        if indicator_masking:
            required_keys += ["advs_indic", "advs_indic_r"]
    else:
        required_keys = ["mals", "advs", "advs_r"]

    # Check that all required tensors are provided
    missing_keys = [key for key in required_keys if key not in tensors]
    if missing_keys:
        raise ValueError(f"Missing required tensors: {missing_keys}")

    # Generate file paths and tensors to save
    file_prefix = model_name
    tensors_to_save = {}

    if is_detector_enabled:
        tensors_to_save.update({
            f"{file_prefix}_mals.pt.gz": tensors["mals_detect"],
            f"{file_prefix}_advs_detect.pt.gz": tensors["advs_detect"],
            f"{file_prefix}_advs_detect_r.pt.gz": tensors["advs_detect_r"],
            f"{file_prefix}_advs_obliv.pt.gz": tensors["advs_obliv"],
            f"{file_prefix}_advs_obliv_r.pt.gz": tensors["advs_obliv_r"],
        })
        if indicator_masking:
            tensors_to_save.update({
                f"{file_prefix}_advs_indic.pt.gz": tensors["advs_indic"],
                f"{file_prefix}_advs_indic_r.pt.gz": tensors["advs_indic_r"],
            })
    else:
        tensors_to_save.update({
            f"{file_prefix}_mals.pt.gz": tensors["mals"],
            f"{file_prefix}_advs.pt.gz": tensors["advs"],
            f"{file_prefix}_advs_r.pt.gz": tensors["advs_r"],
        })

    # Save tensors as compressed .pt.gz files
    for file_name, tensor in tensors_to_save.items():
        file_path = os.path.join(output_dir, file_name)
        with gzip.open(file_path, "wb") as f:
            torch.save(tensor.clone().to(dtype=torch.uint8), f)

    # Package all files into a single zip archive
    zip_file_path = os.path.join(output_dir, f"{model_name}_data.zip")
    with zipfile.ZipFile(zip_file_path, "w") as zipf:
        for file_name in tensors_to_save.keys():
            file_path = os.path.join(output_dir, file_name)
            zipf.write(file_path, arcname=file_name)

    print(f"All files saved and zipped into: {zip_file_path}")

In [ ]:
import torch
import gzip
import zipfile
import os

def load_model_data_from_zip(zip_file_path, output_dir="/content/data"):
    """
    Load tensors from a zipped archive containing compressed .pt.gz files and convert them to float.

    Args:
        zip_file_path (str): Path to the zipped archive (e.g., "model_name_data.zip").
        output_dir (str): Directory to extract the files temporarily.

    Returns:
        dict: A dictionary where keys are file names (without extensions) and values are the loaded tensors as float.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Extract the zip file
    with zipfile.ZipFile(zip_file_path, "r") as zipf:
        zipf.extractall(output_dir)
        print(f"Extracted files to {output_dir}")

    # Load tensors from extracted .pt.gz files and convert to float
    tensors = {}
    for file_name in os.listdir(output_dir):
        if file_name.endswith(".pt.gz"):
            file_path = os.path.join(output_dir, file_name)
            with gzip.open(file_path, "rb") as f:
                tensor_name = os.path.splitext(os.path.splitext(file_name)[0])[0]  # Remove .pt.gz
                tensors[tensor_name] = torch.load(f, weights_only=False).to(dtype=torch.float)  # Convert to float
                print(f"Loaded and converted tensor: {tensor_name}")

    return tensors

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_samples: total success samples in inner loop
# o_success_samples: total success samples

def sigma_zero(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 4,
    initial_const: float = 1.,
    confidence: float = 1e-4,
    confidence2: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Track best perturbations
    best_l0 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)


    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Define outer loop iterations based on detector state and oblivion setting
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):

        if (is_detector_enabled and not oblivion):
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}")
            print("unique values of CONST :", torch.unique(CONST))

        active_inputs = inputs.clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100)

        # Dynamic threshold per feature, and success masks
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)
        success_samples = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs = active_inputs + delta

            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(adv_inputs)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(adv_inputs)
            else:
                logits = model.forward(adv_inputs)

            ben_score, mal_score = logits[:, 0], logits[:, 1]

            # Calculate decision loss
            if is_detector_enabled and not oblivion:
                tau = model.tau[0].item()
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0) + \
                          CONST * torch.clamp(prob - tau + confidence2, min=0.0)
            else:
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)

            # Success mask and update best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)

                true_l0 = delta.data.ne(0).sum(dim=1)  # ne(0) (short for "not equal to zero")
                improved_mask = success_mask & (true_l0 < best_l0)
                best_l0= torch.where(improved_mask, true_l0, best_l0)
                best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)

                success_samples |= success_mask
                o_success_samples |= success_mask

            # L0 approximation
            l0_approx = delta.square() / (delta.square() + sigma) # x^2/(x^2+sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features   # 1/d * l0_approx

            # Combined loss for decision and sparsity
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 500 == 0):
                avg_best_dist = best_l0[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                    f"Loop Success = {success_samples.sum().item():4}/{batch_size} | "
                    f"Total Success: {o_success_samples.sum().item():4}/{batch_size} | "
                    f"Best Distance (avg): {avg_best_dist:6.2f} | "
                    f"Loss: {adv_loss.item():7.2f}"
                )

            # Optimize delta
            optimizer.zero_grad()
            adv_loss.backward()

            # Gradient normalization for stability
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)

            # Update delta and apply scheduler
            optimizer.step()
            scheduler.step()

            # Apply constraints to delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

        # Adjust CONST based on success and updated bounds after the first outer step
        if (outer_step + 1) !=  binary_search_steps:
            upper_bound[success_samples] = torch.minimum(upper_bound[success_samples], CONST[success_samples])
            lower_bound[~success_samples] = torch.maximum(lower_bound[~success_samples], CONST[~success_samples])

            # Step 2: Apply additional masking conditions within active indices
            updated_upper_limit_mask = upper_bound < 1e8
            updated_lower_limit_mask = lower_bound > 0

            CONST[success_samples] /= 100.0
            CONST[~success_samples] *= 100.0

            # Set CONST for active samples that have both updated lower and upper bounds
            CONST[success_samples & updated_lower_limit_mask] = (
                (lower_bound[success_samples & updated_lower_limit_mask] +
                 upper_bound[success_samples & updated_lower_limit_mask]) / 2.
            )
            CONST[~success_samples & updated_upper_limit_mask] = (
                (lower_bound[~success_samples & updated_upper_limit_mask] +
                 upper_bound[~success_samples & updated_upper_limit_mask]) / 2.
            )


    return inputs + best_delta

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

def sigma_zero_rounded(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const: float = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Set outer loop iterations
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-' * 50}\nOuter step: {outer_step + 1}\nUnique values of CONST: {torch.unique(CONST)}")

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Set up confidence, thresholds, and success masks
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples
            adv_inputs = inputs + delta

            # Forward pass
            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(adv_inputs)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(adv_inputs)
            else:
                logits = model.forward(adv_inputs)

            ben_score, mal_score = logits[:, 0], logits[:, 1]

            # Calculate decision loss
            if is_detector_enabled and not oblivion:
                tau = model.tau[0].item()
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0) + \
                          CONST * torch.clamp(prob - tau + confidence2, min=0.0)
            else:
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)

            # Success masks and updating best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)

                    if is_detector_enabled:
                        if model.name == 'KDE':
                            rounded_logits, x_hidden = model.forward_f(adv_inputs_rounded)
                            rounded_prob = model.forward_g(x_hidden, y_pred=0)
                        else:
                            rounded_logits, rounded_prob = model.forward(adv_inputs_rounded)
                    else:
                        rounded_logits = model.forward(adv_inputs_rounded)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)

                    success_samples |= success_mask
                    o_success_samples |= success_mask
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    confidence[success_samples & success_mask_R] -= alpha1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence[success_samples & ~success_mask_R] += alpha1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence[(success_samples & ~success_mask_R) & (confidence > 1)] = 1e-4

                    if is_detector_enabled and not oblivion:
                        confidence2[success_samples & success_mask_R] -= alpha2 * (1 - (scheduler.get_last_lr()[0] / lr))
                        confidence2[success_samples & ~success_mask_R] += alpha2 * (1 - (scheduler.get_last_lr()[0] / lr))
                        confidence2[(success_samples & ~success_mask_R) & (confidence2 > 1)] = 1e-4


            # Calculate L0 approximation and combined loss
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose logging
            if verbose and (step == 0 or (step + 1) % 500 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f}"
                )

            # Backpropagation and optimization
            optimizer.zero_grad()
            adv_loss.backward()

            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)

            optimizer.step()
            scheduler.step()

            # Enforce constraints on delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

        # Update bounds after outer step
        if (outer_step + 1) != binary_search_steps:
            upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
            lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

            # Step 2: Apply additional masking conditions within active indices
            updated_upper_limit_mask = upper_bound < 1e8
            updated_lower_limit_mask = lower_bound > 0

            CONST[success_samples_R] /= 100.0
            CONST[~success_samples_R] *= 100.0

            # Set CONST for active samples that have both updated lower and upper bounds
            CONST[success_samples_R & updated_lower_limit_mask] = (
                (lower_bound[success_samples_R & updated_lower_limit_mask] +
                 upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.
            )
            CONST[~success_samples_R & updated_upper_limit_mask] = (
                (lower_bound[~success_samples_R & updated_upper_limit_mask] +
                 upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.
            )

    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

## Step 2 — The model code and the DetectorTemplate

# Loading Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dropout = 0.6, alpha = 0.2, smooth = False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences)
        gt_labels = torch.cat(gt_labels, dim=0)
        return confidences, gt_labels


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

        if np.any([np.all(y_true == i) for i in range(self.n_classes)]):
            print("class absent.")
            return

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")


    def load(self, model_save_path):
        """
        load model parameters from disk
        """
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading rFGSM
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class AdvMalwareDetectorICNN(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='ICNN', ):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.md_nn_model = md_nn_model
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]


        if hasattr(self.md_nn_model, 'smooth'):
            if not self.md_nn_model.smooth:  # non-smooth, exchange it
                for name, child in self.md_nn_model.named_children():
                    if isinstance(child, nn.ReLU):
                        self.md_nn_model._modules['relu'] = nn.SELU()
        else:
            for name, child in self.md_nn_model.named_children():
                if isinstance(child, nn.ReLU):
                    self.md_nn_model._modules['relu'] = nn.SELU()
        self.md_nn_model = self.md_nn_model.to(self.device)

        # input convex neural network
        self.non_neg_dense_layers = []
        if len(self.dense_hidden_units) < 1:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                                       self.dense_hidden_units[i + 1],
                                                       bias=False))
        self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[-1], 1, bias=False))
        # registration
        for idx_i, dense_layer in enumerate(self.non_neg_dense_layers):
            self.add_module('non_neg_layer_{}'.format(idx_i), dense_layer)

        self.dense_layers = []
        self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        for i in range(len(self.dense_hidden_units[1:])):
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[i]))
        self.dense_layers.append(nn.Linear(self.input_size, 1))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('layer_{}'.format(idx_i), dense_layer)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward_f(self, x):
        return self.md_nn_model(x)

    def forward_g(self, x):
        prev_x = None
        for i, dense_layer in enumerate(self.dense_layers):
            x_add = []
            x1 = dense_layer(x)
            x_add.append(x1)
            if prev_x is not None:
                x2 = self.non_neg_dense_layers[i - 1](prev_x)
                x_add.append(x2)
            prev_x = torch.sum(torch.stack(x_add, dim=0), dim=0)
            if i < len(self.dense_layers):
                prev_x = F.selu(prev_x)
        return prev_x.reshape(-1)

    def forward(self, x):
        return self.forward_f(x), self.forward_g(x)

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)

        rtn_value = (y_pred == 0) & indicator_flag

        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

        return rtn_value

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(torch.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels


    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits_f, logits_g = self.forward(x)
        return torch.softmax(logits_f, dim=-1).detach().cpu().numpy(), logits_g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                x_logits = self.forward_g(x_val)
                probabilities.append(x_logits)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def reset_threshold(self):
        self.tau[0] = 0.


    def load(self, model_save_path):
        # load model
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading PAD
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels



class AMalwareDetectionDLA(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        assert len(self.dense_hidden_units) >= 1, "Expected at least one hidden layer."
        self.alarm_nn_model = TorchAlarm(input_size=sum(self.md_nn_model.dense_hidden_units))

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)

    def forward(self, x):
        hidden_activations = []
        for dense_layer in self.md_nn_model.dense_layers[:-1]:
            x = self.md_nn_model.activation_func(dense_layer(x))
            hidden_activations.append(x)
        logits = self.md_nn_model.dense_layers[-1](x)
        hidden_activations = torch.cat(hidden_activations, dim=-1)
        x_prob = self.alarm_nn_model(hidden_activations).reshape(-1)
        return logits, x_prob

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                print("class absent.")
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)
        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_logits = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1))
                x_prob.append(x_logits)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))


class TorchAlarm(torch.nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(input_size, 112),
            torch.nn.ReLU(),
            torch.nn.Linear(112, 100),
            torch.nn.ReLU(),
            torch.nn.Linear(100, 300),
            torch.nn.ReLU(),
            torch.nn.Linear(300, 200),
            torch.nn.ReLU(),
            torch.nn.Linear(200, 77),
            torch.nn.ReLU(),
            torch.nn.Linear(77, 1),
        ])

    def __call__(self, x, training=False):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels







class AMalwareDetectionDNNPlus(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth

        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   self.n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        self.amd_nn_plus = MalwareDetectionDNN(self.input_size,
                                               self.n_classes + 1,
                                               self.device,
                                               name)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward(self, x):
        logits = self.amd_nn_plus(x)
        logits -= torch.amax(logits, dim=-1, keepdim=True).detach()  # increasing the stability, which might be helpful
        return logits, torch.softmax(logits, dim=-1)[:, -1]

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                print("class absent.")
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)
        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_cent = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1)[:, :2])
                x_prob.append(x_cent)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, g = self.forward(x)
        x_cent = torch.softmax(logits, dim=-1).detach().cpu().numpy()[:, :2]

        return x_cent, g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_cent = self.forward(x_val)
                probabilities.append(x_cent)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

In [ ]:
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class KernelDensityEstimation(nn.Module, DetectorTemplate):
    """
    kernel density estimation upon the penultimate layer

    parameters
    -------------
    @param model, torch.nn.Module, an instantiation of model object
    @param bandwidth, float, variance of Gaussian density function
    @param n_classes, integer, number of classes
    @param ratio, float [0,1], ratio for computing the threshold
    """

    def __init__(self, model, n_centers=1000, bandwidth=20., n_classes=2, ratio=0.95, name='KDE', device='cpu'):
        super(KernelDensityEstimation, self).__init__()
        assert isinstance(model, torch.nn.Module)
        self.model = model
        self.device = device
        self.name = name
        self.n_centers = n_centers
        self.bandwidth = bandwidth
        self.n_classes = n_classes
        self.ratio = ratio
        self.gaussian_means = None

        self.tau = nn.Parameter(torch.zeros([self.n_classes, ], device=self.device), requires_grad=False)

    def forward(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        x_prob = self.forward_g(x, logits.argmax(1).detach())
        return logits, x_prob

    def forward_f(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        return logits, x

    def forward_g(self, x_hidden, y_pred):
        """
        1 / |X_| * \sum(exp(||x - x_||_2^2/sigma^2))

        parameters
        -----------
        @param x_hidden, torch.tensor, node hidden representation
        @param y_pred, torch.tensor, prediction
        """
        size = x_hidden.size()[0]
        dist = [torch.sum(torch.square(means.unsqueeze(dim=0) - x_hidden.unsqueeze(dim=1)), dim=-1) for means in
                self.gaussian_means]
        kd = torch.stack([torch.mean(torch.exp(-d / self.bandwidth ** 2), dim=-1) for d in dist], dim=1)
        # return p(x|y=y_pred)
        return -1 * kd[torch.arange(size), y_pred]

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for density estimation
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        self.eval()
        probabilities = []
        gt_labels = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                logits, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
                gt_labels.append(y_val)
            prob = torch.cat(probabilities, dim=0)
            gt_labels = torch.cat(gt_labels)
            for i in range(self.n_classes):
                prob_x_y = prob[gt_labels == i]
                s, _ = torch.sort(prob_x_y)
                self.tau[i] = s[int((s.shape[0] - 1) * ratio)]

    def predict(self, test_data_producer, indicator_masking=True):
        # evaluation on detector & indicator
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob, y_pred).cpu().numpy()
        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.

        print('The indicator is turning on...')
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"Number of considered samples is {len(y_true)}%")
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")


        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr:.5f}%, False Positive Rate (FPR) is {fpr:.5f}%, F1 score is {f1:.5f}%")


    def eval(self):
        self.model.eval()

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = x.to(device), y.to(device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(F.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)

        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau[y_pred]

    def indicator(self, x_prob, y_pred=None):
        assert y_pred is not None
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.get_tau_sample_wise(y_pred)).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.get_tau_sample_wise(y_pred)
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")
        # res = probability.reshape(-1, 1).repeat_interleave(2, dim=1) >= self.tau
        # return res[torch.arange(res.size()[0]), y_pred]


    def load(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.gaussian_means = ckpt['gaussian_means']
        self.tau = ckpt['tau']
        self.model.load_state_dict(ckpt['base_model'])

## Step 3 — Baseline: the plain DNN under σ

# Attacks

## DNN

### Load DNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

In [ ]:
model_dnn.double().to(device)
model_dnn.predict(test_loader)

In [ ]:
model_dnn.float()

In [ ]:
for x_test, y_test in train_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

In [ ]:
model_dnn(ben_x_batch[:5])

In [ ]:
model_dnn(mal_x_batch[:5])

In [ ]:
model_dnn((mal_x_batch[:5]+0.01))

In [ ]:
model_dnn((mal_x_batch[:5]+0.05))

### Sigma

In [ ]:
mals, advs = attack_mal_batches(
    test_loader=test_loader,
    model=model_dnn,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
ensure_fixed_elements_unchanged(mals, advs, removal_array)

In [ ]:
result, malwares = evaluate_adversarial_examples(mal_x=mals, adv=advs, model=model_dnn, removal_array=removal_array,
                                      is_detector_enabled=False, rounded=False, batch_size=batch_size,norm='L0')

result2, malwares2 = evaluate_adversarial_examples(mal_x=mals, adv=advs, model=model_dnn, removal_array=removal_array,
                                      is_detector_enabled=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
mals_r, advs_r = attack_mal_batches(
    test_loader=test_loader,
    model=model_dnn,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    alpha1= 0.6,
    alpha2= 0.6
)

In [ ]:
ensure_fixed_elements_unchanged(mals_r, advs_r, removal_array)

In [ ]:
result_r, malwares_r = evaluate_adversarial_examples(mal_x=mals_r, adv=advs_r, model=model_dnn, removal_array=removal_array,
                                              is_detector_enabled=False, rounded=False, batch_size=batch_size, norm='L0')

In [ ]:
adv_list = [result,result2,result_r]
labels = ['sigma_zero','sigma_zero_r_outloop','sigma_zero_r_inloop']
mal_x_list = [malwares,malwares2,malwares_r]
plot_adv_accuracies(adv_list, labels, model_dnn, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L0', interval=5)

### Save data

In [ ]:
model_name = model_dnn.name
save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    is_detector_enabled=False,
    mals=mals,
    advs=advs,
    advs_r=advs_r
)

In [ ]:
DNN_tensors = load_model_data_from_zip(zip_file_path="/content/DNN_data.zip", output_dir="/content/data")

In [ ]:
DNN_tensors.keys()

In [ ]:
result_obliv_r2, malwares_obliv_r2 = evaluate_adversarial_examples(mal_x=DNN_tensors['DNN_mals'], adv=DNN_tensors['DNN_advs_r'], model=model_dnn, removal_array=removal_array,
                                              is_detector_enabled=False, rounded=False, batch_size=batch_size, norm='L0')

## Step 4 — DLA: load and check the threshold behaviour

## DLA

### Loading DLA

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dla = AMalwareDetectionDLA(model_dnn,input_size=10000,n_classes=2,device=device,name='DLA')
model_dla.load('/content/amd_dla.pth')
model_dla.to(device).float()

In [ ]:
model_dla.double()
model_dla.predict(test_loader, indicator_masking=True)

In [ ]:
model_dla.float()

In [ ]:
model_dla.tau

In [ ]:
model_dla.tau[0].item()

In [ ]:
model_dla(ben_x_batch[:5])

In [ ]:
model_dla(mal_x_batch[:5])

In [ ]:
model_dla((mal_x_batch[:5]+0.01))

In [ ]:
model_dla((mal_x_batch[:5]+0.05))

In [ ]:
import torch

def calculate_distance(adv_samples, original_samples, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_samples (torch.Tensor): The adversarial samples.
    - original_samples (torch.Tensor): The original samples.
    - norm (str): The norm type ('l0', 'l1', 'l2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")

def sum_of_top_n_distances(adv_rounded, mal_x_batch, y_pred, n, norm='L1'):
    """
    Calculate the sum of the top n smallest distances between adv_rounded and mal_x_batch
    for samples where y_pred == 0, based on the specified norm.

    Parameters:
    - adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
    - mal_x_batch (torch.Tensor): The original malicious examples tensor.
    - y_pred (torch.Tensor): The prediction tensor.
    - n (int): The number of top distances to sum.
    - norm (str): The norm to use ('l0', 'l1', 'l2').

    Returns:
    - torch.Tensor: The sum of the top n smallest distances.
    """
    distances = calculate_distance(adv_rounded[y_pred == 0], mal_x_batch[y_pred == 0], norm)
    top_n_distances = torch.sort(distances).values[:n]
    return top_n_distances.sum()

def evaluate_adversarial_examples(mal_x, adv, model, removal_array, is_detector_enabled=False, oblivion=False, rounded=False, batch_size=1024, norm='L1'):
    """
    Evaluate the model's performance on adversarial examples, optionally applying rounding, in batches if necessary,
    with support for different norms.

    Parameters:
    - mal_x (Tensor): Original malware input.
    - adv (Tensor): Adversarial examples generated from mal_x.
    - model (nn.Module): The trained model to be evaluated.
    - removal_array (Tensor): Array used for final rounding.
    - rounded (bool, optional): Whether to apply rounding on adversarial examples. Default is True.
    - batch_size (int, optional): The maximum number of samples to process at once. Default is 1024.
    - norm (str): The norm to use for distance calculations ('L0', 'L1', or 'L2').

    Returns:
    - final_adv (Tensor): The (rounded) adversarial examples used in evaluation.
    """
    model.eval()
    torch.cuda.empty_cache()

    num_malwares = mal_x.shape[0]
    num_successful_attacks = 0
    total_distances = 0
    y_pred_list = []
    final_adv_list = []
    all_distances = []

    for i in range(0, num_malwares, batch_size):
        mal_x_batch = mal_x[i:i + batch_size]
        adv_batch = adv[i:i + batch_size]

        # Apply rounding if specified
        final_adv_batch = final_rounding(mal_x_batch, adv_batch, model, removal_array, is_detector_enabled, oblivion) if rounded else adv_batch.clone()
        final_adv_list.append(final_adv_batch)

        # Forward pass and calculate successful attacks
        with torch.no_grad():
            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(final_adv_batch)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(final_adv_batch)
                #logits, prob = (model.forward_f(final_adv_batch)[0], model.forward_g(model.forward_f(final_adv_batch)[1], y_pred=0)) if model.name == 'kde' else model.forward(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
            else:
                logits = model.forward(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0)

        y_pred_batch = (~successful_attacks_mask).long()
        y_pred_list.append(y_pred_batch)

        # Calculate and accumulate distances for successful attacks
        num_successful_attacks += successful_attacks_mask.sum().item()
        if successful_attacks_mask.any():
            distances = calculate_distance(final_adv_batch[successful_attacks_mask], mal_x_batch[successful_attacks_mask], norm)
            total_distances += distances.sum().item()
            all_distances.extend(distances.cpu().tolist())

    # Final predictions and adversarial examples aggregation
    y_pred = torch.cat(y_pred_list)
    final_adv = torch.cat(final_adv_list)

    # Calculate evaluation metrics
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100
    mean_distance = total_distances / max(1, num_successful_attacks)
    median_distance = torch.median(torch.tensor(all_distances)).item() if all_distances else 0

    # Print evaluation results
    print(f"Evaluation Results on Adversarial Examples based on Distance:({norm} norm), Rounded:{rounded}")
    print("-----------------------------------------------")
    print(f"Number of successful attacks: {num_successful_attacks}/{num_malwares}")
    print(f"Model accuracy on malware under attack: {accuracy:.2f}%")
    print(f"Mean distance (successful adv. examples vs. original): {mean_distance:.2f}")
    print(f"Median distance (successful adv. examples vs. original): {median_distance:.2f}")

    # Calculate and print mean distances for various TOP N thresholds
    top_n_values = [500, 1000, 2000, 2500, 3000]
    for top_n in top_n_values:
        if num_malwares >= top_n:
            total_distance_top_n = sum_of_top_n_distances(final_adv, mal_x, y_pred, top_n, norm=norm) / top_n
            print(f"Mean distance (successful adv. examples vs. original) - TOP {top_n}: {total_distance_top_n:.2f}")

    return final_adv

## Step 5 — DLA under σ: detector-aware and detector-oblivious

### oblivion=False

In [ ]:
mals_detect, advs_detect = attack_mal_batches(
    test_loader=test_loader,
    model=model_dla,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=3,
    initial_const=0.01,
)

In [ ]:
ensure_fixed_elements_unchanged(mals_detect, advs_detect, removal_array)

In [ ]:
result_detect, malwares_detect = evaluate_adversarial_examples(mal_x=mals_detect, adv=advs_detect, model=model_dla, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size,norm='L0')

result_detect2, malwares_detect2 = evaluate_adversarial_examples(mal_x=mals_detect, adv=advs_detect, model=model_dla, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
mals_detect_r, advs_detect_r = attack_mal_batches(
    test_loader=test_loader,
    model=model_dla,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=3,
    initial_const=0.01,
    alpha1= 0.6,
    alpha2= 0.6
)

In [ ]:
ensure_fixed_elements_unchanged(mals_detect_r, advs_detect_r, removal_array)

In [ ]:
result_detect_r, malwares_detect_r = evaluate_adversarial_examples(mal_x=mals_detect_r, adv=advs_detect_r, model=model_dla, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size, norm='L0')

In [ ]:
adv_list = [result_detect,result_detect2,result_detect_r]
labels = ['sigma_zero','sigma_zero_r_outloop','sigma_zero_r_inloop']
mal_x_list = [malwares_detect,malwares_detect2,malwares_detect_r]
plot_adv_accuracies(adv_list, labels, model_dla, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=5)

### oblivion=True

In [ ]:
# Usage example with specific parameters passed via **kwargs
mals_obliv, advs_obliv = attack_mal_batches(
    test_loader=test_loader,
    model=model_dla,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
ensure_fixed_elements_unchanged(mals_obliv, advs_obliv, removal_array)

In [ ]:
result_obliv, malwares_obliv = evaluate_adversarial_examples(mal_x=mals_obliv, adv=advs_obliv, model=model_dla, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

result_obliv2, malwares_obliv2 = evaluate_adversarial_examples(mal_x=mals_obliv, adv=advs_obliv, model=model_dla, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
# Usage example with specific parameters passed via **kwargs
mals_obliv_r, advs_obliv_r = attack_mal_batches(
    test_loader=test_loader,
    model=model_dla,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf
)

In [ ]:
ensure_fixed_elements_unchanged(mals_obliv_r, advs_obliv_r, removal_array)

In [ ]:
result_obliv_r, malwares_obliv_r = evaluate_adversarial_examples(mal_x=mals_obliv_r, adv=advs_obliv_r, model=model_dla, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

In [ ]:
adv_list = [result_obliv,result_obliv2,result_obliv_r]
labels = ['sigma_zero','sigma_zero_r_outloop','sigma_zero_r_inloop']
mal_x_list = [malwares_obliv,malwares_obliv2,malwares_obliv_r]
plot_adv_accuracies(adv_list, labels, model_dla, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=5, oblivion=True)

### Save data

In [ ]:
model_name = model_dla.name
save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    is_detector_enabled=True,
    indicator_masking=False,
    mals_detect=mals_detect,  # Pass as keyword arguments
    advs_detect=advs_detect,
    advs_detect_r=advs_detect_r,
    advs_obliv=advs_obliv,
    advs_obliv_r=advs_obliv_r
)

In [ ]:
DLA_tensors = load_model_data_from_zip(zip_file_path="/content/DLA_data.zip", output_dir="/content/DLA_data")

In [ ]:
DLA_tensors.keys()

In [ ]:
result_detect_r2, malwares_detect_r2 = evaluate_adversarial_examples(mal_x=DLA_tensors['DLA_mals'], adv=DLA_tensors['DLA_advs_detect_r'], model=model_dla, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

## Step 6 — DNN+: load and check

## DNN_Plus

### Loading DNN_Plus

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dnn_plus = AMalwareDetectionDNNPlus(model_dnn,input_size=10000,n_classes=2,device=device,name='DNNPlus')
model_dnn_plus.load('/content/amd_dnn_plus.pth')
model_dnn_plus.to(device).float()

In [ ]:
model_dnn_plus.double()
model_dnn_plus.predict(test_loader, indicator_masking=True)

In [ ]:
model_dnn_plus.float()

In [ ]:
model_dnn_plus.tau

In [ ]:
model_dnn_plus.tau[0].item()

In [ ]:
model_dnn_plus(ben_x_batch[:5])

In [ ]:
model_dnn_plus(mal_x_batch[:5])

In [ ]:
model_dnn_plus((mal_x_batch[:5]+0.01))

In [ ]:
model_dnn_plus((mal_x_batch[:5]+0.05))

## Step 7 — DNN+ under σ

### oblivion=False

In [ ]:
mals_detect, advs_detect = attack_mal_batches(
    test_loader=test_loader,
    model=model_dnn_plus,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=3,
    initial_const=0.01,
)

In [ ]:
result_detect, malwares_detect = evaluate_adversarial_examples(mal_x=mals_detect, adv=advs_detect, model=model_dnn_plus, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size,norm='L0')

result_detect2, malwares_detect2 = evaluate_adversarial_examples(mal_x=mals_detect, adv=advs_detect, model=model_dnn_plus, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
mals_detect_r, advs_detect_r = attack_mal_batches(
    test_loader=test_loader,
    model=model_dnn_plus,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=3,
    initial_const=0.01,
    alpha1= 0.6,
    alpha2= 0.6
)

In [ ]:
result_detect_r, malwares_detect_r = evaluate_adversarial_examples(mal_x=mals_detect_r, adv=advs_detect_r, model=model_dnn_plus, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size, norm='L0')

In [ ]:
adv_list = [result_detect,result_detect2,result_detect_r]
labels = ['sigma_zero','sigma_zero_r_outloop','sigma_zero_r_inloop']
mal_x_list = [malwares_detect,malwares_detect2,malwares_detect_r]
plot_adv_accuracies(adv_list, labels, model_dnn_plus, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=5)

### oblivion=True

In [ ]:
# Usage example with specific parameters passed via **kwargs
mals_obliv, advs_obliv = attack_mal_batches(
    test_loader=test_loader,
    model=model_dnn_plus,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
result_obliv, malwares_obliv = evaluate_adversarial_examples(mal_x=mals_obliv, adv=advs_obliv, model=model_dnn_plus, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

result_obliv2, malwares_obliv2 = evaluate_adversarial_examples(mal_x=mals_obliv, adv=advs_obliv, model=model_dnn_plus, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
# Usage example with specific parameters passed via **kwargs
mals_obliv_r, advs_obliv_r = attack_mal_batches(
    test_loader=test_loader,
    model=model_dnn_plus,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf
)

In [ ]:
result_obliv_r, malwares_obliv_r = evaluate_adversarial_examples(mal_x=mals_obliv_r, adv=advs_obliv_r, model=model_dnn_plus, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

In [ ]:
adv_list = [result_obliv,result_obliv2,result_obliv_r]
labels = ['sigma_zero','sigma_zero_r_outloop','sigma_zero_r_inloop']
mal_x_list = [malwares_obliv,malwares_obliv2,malwares_obliv_r]
plot_adv_accuracies(adv_list, labels, model_dnn_plus, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=5, oblivion=True)

### Save data

In [ ]:
model_name = model_dnn_plus.name
save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    is_detector_enabled=True,
    indicator_masking=False,
    mals_detect=mals_detect,  # Pass as keyword arguments
    advs_detect=advs_detect,
    advs_detect_r=advs_detect_r,
    advs_obliv=advs_obliv,
    advs_obliv_r=advs_obliv_r
)

In [ ]:
DNNPlus_tensors = load_model_data_from_zip(zip_file_path="/content/DNNPlus_data.zip", output_dir="/content/data_DNNPlus")

In [ ]:
DNNPlus_tensors.keys()

In [ ]:
result_detect_r2, malwares_detect_r2 = evaluate_adversarial_examples(mal_x=DNNPlus_tensors['DNNPlus_mals'], adv=DNNPlus_tensors['DNNPlus_advs_detect_r'], model=model_dnn_plus, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

## Step 8 — ICNN: how it is constructed (from `ICNN.ipynb`)

# New Section

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', **kwargs):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name

        self.parse_args(**kwargs)

        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def parse_args(self,
                   dense_hidden_units=None,
                   dropout=0.6,
                   alpha_=0.2,
                   smooth=False,
                   **kwargs
                   ):
        if dense_hidden_units is None:
            self.dense_hidden_units = [200, 200]
        elif isinstance(dense_hidden_units, list):
            self.dense_hidden_units = dense_hidden_units
        else:
            raise TypeError("Expect a list of hidden units.")

        self.dropout = dropout
        self.alpha_ = alpha_
        self.smooth = smooth
        self.proc_number = kwargs['proc_number']


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError

def to_tensor(feature_x=None, labels=None, device='cpu'):
    """Convert features, labels from array or sparse matrix to
    torch Tensor.
    code is adapted from: https://github.com/deeprobust/DeepRobust/graph/utils.py
    Parameters
    ----------
    adj : scipy.sparse.csr_matrix
        the adjacency matrix.
    features : scipy.sparse.csr_matrix
        node features
    labels : numpy.array
        node labels
    device : str
        'cpu' or 'cuda'
    """

    def _to_torch_tensor(mat):
        if sp.issparse(mat):
            mat = sparse_mx_to_torch_sparse_tensor(mat)
        elif isinstance(mat, torch.Tensor):
            pass
        else:
            mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


def psn(x_tensor, prob, lower_value=0., upper_value=1.):
    assert 1. >= prob >= 0.
    uni_noises = torch.DoubleTensor(x_tensor.shape).uniform_()
    salt_pos = uni_noises >= prob
    uni_noises[salt_pos] = upper_value
    uni_noises[~salt_pos] = lower_value
    if x_tensor.get_device() >= 0:
        return uni_noises.to(x_tensor.get_device())
    else:
        return uni_noises

class AdvMalwareDetectorICNN(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', **kwargs):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.parse_args(**kwargs)
        if isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
        else:
            kwargs['smooth'] = True
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   n_classes,
                                                   self.device,
                                                   name,
                                                   **kwargs)
            warnings.warn("Use a self-defined NN-based malware detector")
        if hasattr(self.md_nn_model, 'smooth'):
            if not self.md_nn_model.smooth:  # non-smooth, exchange it
                for name, child in self.md_nn_model.named_children():
                    if isinstance(child, nn.ReLU):
                        self.md_nn_model._modules['relu'] = nn.SELU()
        else:
            for name, child in self.md_nn_model.named_children():
                if isinstance(child, nn.ReLU):
                    self.md_nn_model._modules['relu'] = nn.SELU()
        self.md_nn_model = self.md_nn_model.to(self.device)

        # input convex neural network
        self.non_neg_dense_layers = []
        if len(self.dense_hidden_units) < 1:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                                       self.dense_hidden_units[i + 1],
                                                       bias=False))
        self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[-1], 1, bias=False))
        # registration
        for idx_i, dense_layer in enumerate(self.non_neg_dense_layers):
            self.add_module('non_neg_layer_{}'.format(idx_i), dense_layer)

        self.dense_layers = []
        self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        for i in range(len(self.dense_hidden_units[1:])):
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[i]))
        self.dense_layers.append(nn.Linear(self.input_size, 1))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('layer_{}'.format(idx_i), dense_layer)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def parse_args(self,
                   dense_hidden_units=None,
                   dropout=0.6,
                   alpha_=0.2,
                   **kwargs
                   ):
        if dense_hidden_units is None:
            self.dense_hidden_units = [200, 200]
        elif isinstance(dense_hidden_units, list):
            self.dense_hidden_units = dense_hidden_units
        else:
            raise TypeError("Expect a list of hidden units.")

        self.dropout = dropout
        self.alpha_ = alpha_
        self.proc_number = kwargs['proc_number']

    def forward_f(self, x):
        return self.md_nn_model(x)

    def forward_g(self, x):
        prev_x = None
        for i, dense_layer in enumerate(self.dense_layers):
            x_add = []
            x1 = dense_layer(x)
            x_add.append(x1)
            if prev_x is not None:
                x2 = self.non_neg_dense_layers[i - 1](prev_x)
                x_add.append(x2)
            prev_x = torch.sum(torch.stack(x_add, dim=0), dim=0)
            if i < len(self.dense_layers):
                prev_x = F.selu(prev_x)
        return prev_x.reshape(-1)

    def forward(self, x):
        return self.forward_f(x), self.forward_g(x)

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)

        rtn_value = (y_pred == 0) & indicator_flag

        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

        return rtn_value

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(torch.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels


    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits_f, logits_g = self.forward(x)
        return torch.softmax(logits_f, dim=-1).detach().cpu().numpy(), logits_g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                x_logits = self.forward_g(x_val)
                probabilities.append(x_logits)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def reset_threshold(self):
        self.tau[0] = 0.

    def customize_loss(self, logits_x, labels, logits_adv_x, labels_adv, beta_1=1, beta_2=1):
        if logits_adv_x is not None and len(logits_adv_x) > 0:
            G = F.binary_cross_entropy_with_logits(logits_adv_x, labels_adv)
        else:
            G = 0
        if logits_x is not None and len(logits_x) > 0:
            F_ = F.cross_entropy(logits_x, labels)
        else:
            F_ = 0
        return beta_1 * F_ + beta_2 * G

    def fit(self, train_data_producer, validation_data_producer, epochs=100, lr=0.005, weight_decay=0., verbose=True):
        """
        Train the malware & adversary detector, pick the best model according to the validation results

        Parameters
        ----------
        @param train_data_producer: Object, an iterator for producing a batch of training data
        @param validation_data_producer: Object, an iterator for producing validation dataset
        @param epochs, Integer, epochs
        @param lr, Float, learning rate for Adam optimizer
        @param weight_decay, Float, penalty factor
        @param verbose: Boolean, whether to show verbose logs
        """
        optimizer = optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        best_avg_acc = 0.
        best_epoch = 0
        total_time = 0.
        nbatches = len(train_data_producer)
        for i in range(epochs):
            self.train()
            losses, accuracies = [], []
            for idx_batch, (x_train, y_train) in enumerate(train_data_producer):
                x_train, y_train = to_device(x_train.double(), y_train.long(), self.device)
                # make data for training g
                # 1. add pepper and salt noises
                x_train_noises = torch.clamp(x_train + psn(x_train, np.random.uniform(0, 0.5)),
                                             min=0., max=1.)
                x_train_ = torch.cat([x_train, x_train_noises], dim=0)
                y_train_ = torch.cat([torch.zeros(x_train.shape[:1]), torch.ones(x_train.shape[:1])]).double().to(
                    self.device)
                idx = torch.randperm(y_train_.shape[0])
                x_train_ = x_train_[idx]
                y_train_ = y_train_[idx]

                start_time = time.time()
                optimizer.zero_grad()
                logits_f = self.forward_f(x_train)
                logits_g = self.forward_g(x_train_)
                loss_train = self.customize_loss(logits_f, y_train, logits_g, y_train_)
                loss_train.backward()
                optimizer.step()
                # clamp
                constraint = utils.NonnegWeightConstraint()
                for name, module in self.named_modules():
                    if 'non_neg_layer' in name:
                        module.apply(constraint)
                total_time = total_time + time.time() - start_time
                acc_f_train = (logits_f.argmax(1) == y_train).sum().item()
                acc_f_train /= x_train.size()[0]
                acc_g_train = ((F.sigmoid(logits_g) >= 0.5) == y_train_).sum().item()
                acc_g_train /= x_train_.size()[0]

                mins, secs = int(total_time / 60), int(total_time % 60)
                losses.append(loss_train.item())
                accuracies.append(acc_f_train)
                accuracies.append(acc_g_train)
                if verbose:
                    logger.info(
                        f'Mini batch: {i * nbatches + idx_batch + 1}/{epochs * nbatches} | training time in {mins:.0f} minutes, {secs} seconds.')
                    logger.info(
                        f'Training loss (batch level): {losses[-1]:.4f} | Train accuracy: {acc_f_train * 100:.2f}% & {acc_g_train * 100:.2f}%.')

            self.eval()
            avg_acc_val = []
            with torch.no_grad():
                for x_val, y_val in validation_data_producer:
                    x_val, y_val = to_device(x_val.double(), y_val.long(), self.device)
                    x_val_noises = torch.clamp(x_val + psn(x_val, np.random.uniform(0, 0.5)),
                                               min=0., max=1.)
                    x_val_ = torch.cat([x_val, x_val_noises], dim=0)
                    y_val_ = torch.cat([torch.zeros(x_val.shape[:1]), torch.ones(x_val.shape[:1])]).long().to(
                        self.device)
                    logits_f = self.forward_f(x_val)
                    logits_g = self.forward_g(x_val_)
                    acc_val = (logits_f.argmax(1) == y_val).sum().item()
                    acc_val /= x_val.size()[0]
                    avg_acc_val.append(acc_val)

                    acc_val_g = ((F.sigmoid(logits_g) >= 0.5) == y_val_).sum().item()
                    acc_val_g /= x_val_.size()[0]
                    avg_acc_val.append(acc_val_g)
                avg_acc_val = np.mean(avg_acc_val)

            if avg_acc_val >= best_avg_acc:
                best_avg_acc = avg_acc_val
                best_epoch = i
                self.get_threshold(validation_data_producer)

                self.save_to_disk()
                if verbose:
                    print(f'Model saved at path: {self.model_save_path}')

            if verbose:
                logger.info(
                    f'Training loss (epoch level): {np.mean(losses):.4f} | Train accuracy: {np.mean(accuracies) * 100:.2f}')
                logger.info(
                    f'Validation accuracy: {avg_acc_val * 100:.2f} | The best validation accuracy: {best_avg_acc * 100:.2f} at epoch: {best_epoch}')

    def load(self):
        # load model
        assert path.exists(self.model_save_path), 'train model first'
        # ckpt = torch.load(self.model_save_path)
        # self.tau = ckpt['tau']
        # self.md_nn_model.load_state_dict(ckpt['md_model'])
        self.load_state_dict(torch.load(self.model_save_path))

    def save_to_disk(self):
        if not path.exists(self.model_save_path):
            utils.mkdir(path.dirname(self.model_save_path))
        # torch.save({
        #     'tau': self.tau,
        #     'md_model': self.md_nn_model.state_dict(),
        #     'amd_model': self.state_dict()
        # }, self.model_save_path
        # )
        torch.save(self.state_dict(), self.model_save_path)

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device,proc_number = 10)
model_dnn.load_state_dict(torch.load('/content/model_dnn.pth', map_location=torch.device(device)))
model_dnn.to(device)

In [ ]:
ICNN = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,proc_number = 10)
ICNN.load_state_dict(torch.load('/content/amd_icnn.pth', map_location=torch.device(device)))
ICNN.to(device).float()

# New Section

In [ ]:
ICNN.predict(test_loader, indicator_masking=True)

In [ ]:
ICNN.tau

In [ ]:
ICNN.tau[0].item()

In [ ]:
ICNN(ben_x_batch[:5])

In [ ]:
ICNN(mal_x_batch[:5])

In [ ]:
ICNN((mal_x_batch[:5]+0.01))

In [ ]:
ICNN((mal_x_batch[:5]+0.05))

## Step 9 — ICNN under σ

## ICNN

### Loadin ICNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_icnn = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='ICNN')
model_icnn.load('/content/amd_icnn.pth')
model_icnn.to(device).float()

In [ ]:
model_icnn.double().to(device)
model_icnn.predict(test_loader, indicator_masking=True)

In [ ]:
model_icnn.to(device).float()

In [ ]:
model_icnn.tau

In [ ]:
model_icnn.tau[0].item()

In [ ]:
model_icnn(ben_x_batch[:5])

In [ ]:
model_icnn(mal_x_batch[:5])

In [ ]:
model_icnn((mal_x_batch[:5]+0.01))

In [ ]:
model_icnn((mal_x_batch[:5]+0.05))

### oblivion=False

In [ ]:
mals_detect, advs_detect = attack_mal_batches(
    test_loader=test_loader,
    model=model_icnn,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=3,
    initial_const=0.01,
)

In [ ]:
result_detect, malwares_detect = evaluate_adversarial_examples(mal_x=mals_detect, adv=advs_detect, model=model_icnn, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size,norm='L0')

result_detect2, malwares_detect2 = evaluate_adversarial_examples(mal_x=mals_detect, adv=advs_detect, model=model_icnn, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
mals_detect_r, advs_detect_r = attack_mal_batches(
    test_loader=test_loader,
    model=model_icnn,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=3,
    initial_const=0.01,
    alpha1= 0.6,
    alpha2= 0.6
)

In [ ]:
result_detect_r, malwares_detect_r = evaluate_adversarial_examples(mal_x=mals_detect_r, adv=advs_detect_r, model=model_icnn, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size, norm='L0')

In [ ]:
adv_list = [result_detect,result_detect2,result_detect_r]
labels = ['sigma_zero','sigma_zero_r_outloop','sigma_zero_r_inloop']
mal_x_list = [malwares_detect,malwares_detect2,malwares_detect_r]
plot_adv_accuracies(adv_list, labels, model_icnn, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=5)

### oblivion=True

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# Usage example with specific parameters passed via **kwargs
mals_obliv, advs_obliv = attack_mal_batches(
    test_loader=test_loader,
    model=model_icnn,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
result_obliv, malwares_obliv = evaluate_adversarial_examples(mal_x=mals_obliv, adv=advs_obliv, model=model_icnn, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

result_obliv2, malwares_obliv2 = evaluate_adversarial_examples(mal_x=mals_obliv, adv=advs_obliv, model=model_icnn, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
# Usage example with specific parameters passed via **kwargs
mals_obliv_r, advs_obliv_r = attack_mal_batches(
    test_loader=test_loader,
    model=model_icnn,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=0.6,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf
)

In [ ]:
result_obliv_r, malwares_obliv_r = evaluate_adversarial_examples(mal_x=mals_obliv_r, adv=advs_obliv_r, model=model_icnn, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

In [ ]:
adv_list = [result_obliv,result_obliv2,result_obliv_r]
labels = ['sigma_zero','sigma_zero_r_outloop','sigma_zero_r_inloop']
mal_x_list = [malwares_obliv,malwares_obliv2,malwares_obliv_r]
plot_adv_accuracies(adv_list, labels, model_icnn, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=5, oblivion=True)

### Save data

In [ ]:
model_name = model_icnn.name
save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    is_detector_enabled=True,
    indicator_masking=False,
    mals_detect=mals_detect,  # Pass as keyword arguments
    advs_detect=advs_detect,
    advs_detect_r=advs_detect_r,
    advs_obliv=advs_obliv,
    advs_obliv_r=advs_obliv_r
)

In [ ]:
ICNN_tensors = load_model_data_from_zip(zip_file_path="/content/ICNN_data.zip", output_dir="/content/data_ICNN")

In [ ]:
ICNN_tensors.keys()

In [ ]:
result_detect_r2, malwares_detect_r2 = evaluate_adversarial_examples(mal_x=ICNN_tensors['ICNN_mals'], adv=ICNN_tensors['ICNN_advs_detect_r'], model=model_icnn, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size, norm='L0')

## Step 10 — PAD: checks and the σ attack in detail (from `ICNN.ipynb`)

In [ ]:
PAD = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,proc_number = 10)
PAD.load_state_dict(torch.load('/content/amd_pad_ma.pth', map_location=torch.device(device)))
PAD.to(device).float()

In [ ]:
PAD = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,proc_number = 10)
ckpt = torch.load('/content/amd_pad_ma.pth')
PAD.load_state_dict(ckpt['model'])
PAD.to(device).float()

In [ ]:
PAD.to(device).double ()
PAD.predict(test_loader, indicator_masking=True)

In [ ]:
PAD.tau

In [ ]:
PAD.tau[0].item()

In [ ]:
PAD(ben_x_batch[:5])

In [ ]:
PAD(mal_x_batch[:5])

In [ ]:
PAD((mal_x_batch[:5]+0.01))

In [ ]:
PAD((mal_x_batch[:5]+0.001))

# sigma_zero

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

def sigma_zero(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 100,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 8,
    initial_const: float = 1.,
    confidence: float = 1e-4,
    confidence2: float = 1e-4,
    device: str = 'cpu'
):
    # Function to apply a mask on gradients
    def grad_hook(grad):
        return grad * non_fixed_features_mask[active_mask]


    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)).bool()

    # Track best perturbations
    best_l0 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)


    # Initialize CONST and bounds
    CONST = torch.zeros(batch_size, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Define outer loop iterations based on detector state and oblivion setting
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

    for outer_step in range(outer_iters):
        print(f'------------------------------------------------------------------------------------------------------------------------------------')
        print(f'Outer step: {outer_step + 1}')

        if outer_step == 1:
            # Mask unsuccessful samples for further processing
            active_mask = (~o_success_mask).clone().to(device)
            if active_mask.sum() == 0:
                print("All samples succeeded. Exiting early.")
                break

            # Reset CONST only for unsuccessful samples
            CONST[active_mask] = initial_const

        # Adjust CONST based on success and updated bounds after the first outer step
        if outer_step > 1:
            # Step 1: Get indices where `active_mask` is True
            active_indices = active_mask.nonzero(as_tuple=True)[0]

            # Step 2: Apply additional masking conditions within active indices
            updated_upper_limit_mask = upper_bound[active_indices] < 1e8
            updated_lower_limit_mask = lower_bound[active_indices] > 0

            # Apply updates only to CONST values that meet specified conditions
            successful_active_indices = active_indices[success_mask]
            unsuccessful_active_indices = active_indices[~success_mask & ~updated_upper_limit_mask]

            CONST[successful_active_indices] /= 10.0
            CONST[unsuccessful_active_indices] *= 10.0

            # Set CONST for active samples that have both updated lower and upper bounds
            CONST[active_indices[success_mask & updated_lower_limit_mask]] = (
                (lower_bound[active_indices[success_mask & updated_lower_limit_mask]] +
                 upper_bound[active_indices[success_mask & updated_lower_limit_mask]]) / 2.
            )
            CONST[active_indices[~success_mask & updated_upper_limit_mask]] = (
                (lower_bound[active_indices[~success_mask & updated_upper_limit_mask]] +
                 upper_bound[active_indices[~success_mask & updated_upper_limit_mask]]) / 2.
            )


        print("CONST values:", CONST[active_mask])
        active_inputs = inputs[active_mask].clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(grad_hook)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 10)

        # Dynamic threshold per feature
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)

        success_mask = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs = active_inputs + delta

            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(adv_inputs)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(adv_inputs)
            else:
                logits = model.forward(adv_inputs)

            ben_score = logits[:, 0]
            mal_score = logits[:, 1]

            if is_detector_enabled and not oblivion:
                tau = model.tau[0].item()
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0) + CONST[active_mask] * torch.clamp(prob - tau  + confidence2, min=0.0)
                #dl_loss = (mal_score - ben_score) + CONST[active_mask] * torch.clamp(prob - tau + kappa, min=0.0)
            else:
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)

            # L0 approximation
            l0_approx = delta.square() / (delta.square() + sigma) # x^2/(x^2+sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features   # 1/d * l0_approx

            # Success mask and update best solutions
            with torch.no_grad():

                if is_detector_enabled and not oblivion:
                    success_mask_inloop = (logits.argmax(dim=1) == 0) & (prob <= tau)
                else:
                    success_mask_inloop = (logits.argmax(dim=1) == 0)

                true_l0 = delta.data.ne(0).sum(dim=1)  # ne(0) (short for "not equal to zero")
                improved_mask = success_mask_inloop & (true_l0 < best_l0[active_mask])

                best_l0[active_mask] = torch.where(improved_mask, true_l0, best_l0[active_mask])
                best_delta[active_mask] = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta[active_mask])
                success_mask |= success_mask_inloop
                o_success_mask[active_mask] |= success_mask_inloop

            # Combined loss for decision and sparsity
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 200 == 0):
                print(f"Iteration {step + 1:5}:\tLR = {scheduler.get_last_lr()[0]:.2f}\t"
                      f"Success = {success_mask_inloop.sum().item()}/{batch_size}\t"
                      f"Total Success = {o_success_mask.sum().item()}\t"
                      f"Best L0 (avg) = {best_l0[o_success_mask].float().mean().item():.2f}\t"
                      f"Loss = {adv_loss.item():.2f}")

            # Optimize delta
            optimizer.zero_grad()
            adv_loss.backward()

            # Gradient normalization for stability
            delta_grad = delta.grad.flatten(1)
            grad_norms = delta_grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            delta.grad = (delta_grad / grad_norms).view_as(delta.grad)

            # Update delta and apply scheduler
            optimizer.step()
            scheduler.step()

            # Apply constraints to delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask_inloop] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask_inloop] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

    return inputs + best_delta

In [ ]:
for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      print(mal_x_batch.shape)
      break

In [ ]:
adv = sigma_zero(model=PAD, inputs=mal_x_batch[:100], removal_array=removal_array, steps=1000, lr=1.0, sigma=1e-4, threshold=0.3, verbose=True, grad_norm=torch.inf, device=device)

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

def sigma_zero_rounded(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const: float = 1.,
    device: str = 'cpu'
):
    # Function to apply a mask on gradients
    def grad_hook(grad):
        return grad * non_fixed_features_mask[active_mask]

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)).bool()

    # Track best perturbations
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)


    # Initialize CONST and bounds
    CONST = torch.zeros(batch_size, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Define outer loop iterations based on detector state and oblivion setting
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

    for outer_step in range(outer_iters):
        print(f'------------------------------------------------------------------------------------------------------------------------------------')
        print(f'Outer step: {outer_step + 1}')

        if outer_step == 1:
            # Mask unsuccessful samples for further processing
            active_mask = (~o_success_samples).clone().to(device)
            if active_mask.sum() == 0:
                print("All samples succeeded. Exiting early.")
                break

            # Reset CONST only for unsuccessful samples
            CONST[active_mask] = initial_const

        # Adjust CONST based on success and updated bounds after the first outer step
        if outer_step > 1:
            # Step 1: Get indices where `active_mask` is True
            active_indices = active_mask.nonzero(as_tuple=True)[0]

            upper_bound[active_indices[success_samples]] = torch.minimum(upper_bound[active_indices[success_samples]], CONST[active_indices[success_samples]])
            lower_bound[active_indices[~success_samples]] = torch.maximum(lower_bound[active_indices[~success_samples]], CONST[active_indices[~success_samples]])

            # Step 2: Apply additional masking conditions within active indices
            updated_upper_limit_mask = upper_bound[active_mask] < 1e8
            updated_lower_limit_mask = lower_bound[active_mask] > 0

            # Apply updates only to CONST values that meet specified conditions
            CONST[active_indices[success_samples & ~updated_lower_limit_mask]] /= 10.0
            CONST[active_indices[~success_samples & ~updated_upper_limit_mask]] *= 10.0

            CONST[active_indices[success_samples & updated_lower_limit_mask]] = (
                (lower_bound[active_indices[success_samples & updated_lower_limit_mask]] +
                 upper_bound[active_indices[success_samples & updated_lower_limit_mask]]) / 2.
            )
            CONST[active_indices[~success_samples & updated_upper_limit_mask]] = (
                (lower_bound[active_indices[~success_samples & updated_upper_limit_mask]] +
                 upper_bound[active_indices[~success_samples & updated_upper_limit_mask]]) / 2.
            )


        print("CONST values:", CONST[active_mask][:10])
        active_inputs = inputs[active_mask].clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(grad_hook)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 10)

        # Dynamic threshold per feature
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)

        confidence = torch.ones(active_inputs.shape[0], device=device) * 1e-4
        confidence2 = torch.ones(active_inputs.shape[0], device=device) * 1e-4

        success_samples = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs = active_inputs + delta

            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(adv_inputs)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(adv_inputs)
            else:
                logits = model.forward(adv_inputs)

            ben_score = logits[:, 0]
            mal_score = logits[:, 1]

            if is_detector_enabled and not oblivion:
                tau = model.tau[0].item()
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0) + CONST[active_mask] * torch.clamp(prob - tau  + confidence2, min=0.0)
                #dl_loss = (mal_score - ben_score) + CONST[active_mask] * torch.clamp(prob - tau + kappa, min=0.0)
            else:
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)


            # Success mask and update best solutions
            with torch.no_grad():

                if is_detector_enabled and not oblivion:
                    success_mask_not_rounded = (logits.argmax(dim=1) == 0) & (prob <= tau)
                else:
                    success_mask_not_rounded = (logits.argmax(dim=1) == 0)

                if step == 0 or (step + 1) % 25 == 0:

                    adv_inputs_rounded = final_rounding(active_inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)

                    if is_detector_enabled:
                        if model.name == 'kde':
                            logits, x_hidden = model.forward_f(adv_inputs_rounded)
                            prob = model.forward_g(x_hidden, y_pred=0)
                        else:
                            rounded_logits, prob = model.forward(adv_inputs_rounded)
                    else:
                        rounded_logits = model.forward(adv_inputs_rounded)

                    if is_detector_enabled and not oblivion:
                        success_mask = (rounded_logits.argmax(dim=1) == 0) & (prob <= tau)
                    else:
                        success_mask = (rounded_logits.argmax(dim=1) == 0)

                    # Calculate L0 distance for rounded inputs
                    dist = torch.abs(active_inputs - adv_inputs_rounded).sum(dim=1)
                    improved_mask = success_mask & (dist < best_dist[active_mask])

                    # Update best perturbations for successful samples
                    best_dist[active_mask] = torch.where(improved_mask, dist, best_dist[active_mask])
                    best_delta[active_mask] = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta[active_mask])
                    success_samples |= success_mask
                    o_success_samples[active_mask] |= success_mask

                    confidence[success_mask_not_rounded & success_mask] -= 0.05
                    confidence[success_mask_not_rounded & ~success_mask] += 0.1
                    confidence[~success_mask_not_rounded & ~success_mask] = 1e-4

                    confidence2[success_mask_not_rounded & success_mask] -= 0.05
                    confidence2[success_mask_not_rounded & ~success_mask] += 0.1
                    confidence2[~success_mask_not_rounded & ~success_mask] = 1e-4



            # L0 approximation
            l0_approx = delta.square() / (delta.square() + sigma) # x^2/(x^2+sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features   # 1/d * l0_approx

            # Combined loss for decision and sparsity
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 200 == 0):
                avg_best_dist = best_dist[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(f"Iteration {step + 1:5} | LR = {scheduler.get_last_lr()[0]:.2f} |Current Success = {success_mask.sum().item()}/{active_mask.sum().item()} |Total Success = {o_success_samples.sum().item()}/{batch_size} | Best Dist (avg) = {avg_best_dist:.2f} | Loss = {adv_loss.item():.2f}")

            # Optimize delta
            optimizer.zero_grad()
            adv_loss.backward()

            # Gradient normalization for stability
            delta_grad = delta.grad.flatten(1)
            grad_norms = delta_grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            delta.grad = (delta_grad / grad_norms).view_as(delta.grad)

            # Update delta and apply scheduler
            optimizer.step()
            scheduler.step()

            # Apply constraints to delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask_not_rounded] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask_not_rounded] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

    return inputs + best_delta

In [ ]:
adv = sigma_zero_rounded(model=ICNN, inputs=mal_x_batch[:3], removal_array=removal_array, steps=1000, lr=1.0, sigma=1e-4, threshold=0.3, verbose=True, grad_norm=torch.inf, device=device)

In [ ]:
# Usage example with specific parameters passed via **kwargs
combined_mals, combined_advs = attack_mal_batches(
    test_loader=test_loader,
    model=PAD,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=5000,
    lr=1.0,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=5,
    initial_const=0.1
)

In [ ]:
ensure_fixed_elements_unchanged(combined_mals, combined_advs, removal_array)

In [ ]:
result = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=ICNN, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size)

result2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=ICNN, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size)

In [ ]:
# Usage example with specific parameters passed via **kwargs
combined_mals, combined_advs_round = attack_mal_batches(
    test_loader=test_loader,
    model=ICNN,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=False,
    steps=1000,
    lr=1.0,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=5,
    initial_const=0.1
)

In [ ]:
ensure_fixed_elements_unchanged(combined_mals, combined_advs_round, removal_array)

In [ ]:
result_rounded = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=ICNN, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=False, batch_size=batch_size)

result_rounded2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=ICNN, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size)

In [ ]:
result_rounded3 = evaluate_adversarial_examples(mal_x=combined_mals, adv=result_rounded2, model=ICNN, removal_array=removal_array,
                                              is_detector_enabled=True, rounded=True, batch_size=batch_size)

In [ ]:
adv_rounded_list = [result,result2,result_rounded2]
labels = ['sigma_zero','sigma_zero_r','sigma_zero_rounded']
plot_adv_distances(adv_rounded_list, labels, ICNN, combined_mals, is_detector_enabled=True, device=device)

## oblivion=True

In [ ]:
# Usage example with specific parameters passed via **kwargs
combined_mals, combined_advs = attack_mal_batches(
    test_loader=test_loader,
    model=ICNN,
    attack=sigma_zero,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=1.0,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=5,
    initial_const=0.1
)

In [ ]:
result = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=ICNN, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size)

result2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=ICNN, removal_array=removal_array,
                                      is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size)

In [ ]:
# Usage example with specific parameters passed via **kwargs
combined_mals, combined_advs_round = attack_mal_batches(
    test_loader=test_loader,
    model=ICNN,
    attack=sigma_zero_rounded,
    removal_array=removal_array,
    device=device,
    is_detector_enabled=True,
    oblivion=True,
    steps=1000,
    lr=1.0,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps=5,
    initial_const=0.1
)

In [ ]:
result_rounded = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=ICNN, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=False, batch_size=batch_size)

result_rounded2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs_round, model=ICNN, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size)

In [ ]:
result_rounded3 = evaluate_adversarial_examples(mal_x=combined_mals, adv=result_rounded2, model=ICNN, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=True, rounded=True, batch_size=batch_size)